In [2]:
from dash import Dash, dcc, html, Input, Output, callback
import plotly.express as px
import datetime
import pandas as pd
import geopandas as gpd 
import json

In [3]:
df = pd.read_csv("../data/data_cleaned/patients_adults_paris/patients_FR_geocoded_adulte_clinique.csv",sep=";",dtype={'codepost':str}).drop("Unnamed: 0",axis=1)
# id_dcd = pd.read_excel('../data/data_octobre_2023/dcd_pseudo.xlsx')
# dcd = id_dcd['pseudo_provisoire'].to_list()

# df = df[~df['pseudo_provisoire'].isin(dcd)]
#df.to_csv("../data/data_cleaned/patients_adults_paris/patients_FR_geocoded_adulte_clinique.csv",sep=";")#.drop("Unnamed: 0",axis=1)



In [27]:
len(df)

57878

In [18]:
##Lecture des données patients 
gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y)).set_crs(epsg=4326))

##Lecture données géospatiales 
df_iris = gpd.read_file('H:/canc_air/data/zones_geographiques/iris/CONTOURS-IRIS.shp')
df_epci = gpd.read_file('H:/canc_air/data/zones_geographiques/epci/EPCI_SHAPEFILE.shp')
df_dept = gpd.read_file('H:/canc_air/data/zones_geographiques/departements/DEPARTEMENT.shp', dtype = {'CODE_DEPT':str})
df_region = gpd.read_file('H:/canc_air/data/zones_geographiques/region/REGION.shp')

# ##Lecture des données géospatiales en format geojson 
# with open("H:/canc_air/data/zones_geographiques/departements/DEPARTEMENT.geojson") as f:
#         dept_geojson = json.load(f)

# with open("H:/canc_air/data/zones_geographiques/epci/EPCI.geojson") as f:
#         epci_geojson = json.load(f)

# with open("H:/canc_air/data/zones_geographiques/iris/iris.geojson") as f:
#         iris_geojson = json.load(f)

# with open('H:/canc_air/data/zones_geographiques/region/region.geojson') as f:
#         region_geojson = json.load(f)

## jointure population 

In [23]:
df_pop = pd.read_csv("C:/Users/jbocque1/Downloads/ensemble/donnees_departements.csv",sep=";",dtype = {'CODDEP':str})
df_pop = df_pop[['CODREG','CODDEP','DEP','PTOT']]
df_dept = df_dept[['CODE_DEPT','NOM_DEPT','CODE_REG','NOM_REG','geometry']]

# df_dept['CODE_DEPT'] = df_dept['CODE_DEPT'].astype(str)
# df_pop['CODREG'] = df_pop['CODREG'].astype(str)


In [29]:
df_dept_pop = df_dept.merge(df_pop[['CODDEP','PTOT']], left_on = 'CODE_DEPT', right_on ='CODDEP')

In [26]:
def mise_en_forme_figure(gdf, df_spatiale, CODE_ZONE): 

    patient_counts = gdf[CODE_ZONE].value_counts().reset_index()
    patient_counts.columns = [CODE_ZONE,'patient_count']

    if CODE_ZONE == 'CODE_EPCI' or CODE_ZONE == "INSEE_REG":
        patient_counts[CODE_ZONE]= patient_counts[CODE_ZONE].astype(int)
        df_spatiale[CODE_ZONE]= df_spatiale[CODE_ZONE].astype(int)
    else : 
        patient_counts[CODE_ZONE]= patient_counts[CODE_ZONE].astype(str)
        df_spatiale[CODE_ZONE]= df_spatiale[CODE_ZONE].astype(str)

    spatial_with_patients = patient_counts.merge(df_spatiale, on=CODE_ZONE, how='right')
    spatial_with_patients['patient_count'] = spatial_with_patients['patient_count'].fillna(0)


    ##CODE_ZONE == "CODE_IRIS"
    #spatial_with_patients_simplified = spatial_with_patients[[CODE_ZONE,'patient_count', 'geometry']] # "INSEE_REG","CODE_DEPT","CODE_IRIS",

    #spatial_with_patients_simplified['patient_count'] = spatial_with_patients_simplified['patient_count'].fillna(0)

    ##CODE_ZONE == "CODE_DEPT"
    #spatial_with_patients_simplified = spatial_with_patients[[CODE_ZONE,"CODE_REG","NOM_DEPT",'patient_count', 'geometry']] # "INSEE_REG","CODE_DEPT","CODE_IRIS",
    
    ##CODE_ZONE =="INSEE_REG"
    spatial_with_patients_simplified = spatial_with_patients[[CODE_ZONE,'NOM_DEPT',"CODE_REG",'NOM_REG','PTOT','patient_count', 'geometry']] # "INSEE_REG","CODE_DEPT","CODE_IRIS",

    

    return patient_counts, spatial_with_patients_simplified

In [31]:
value = "Departement"
if value =="Departement":
    CODE_ZONE = "CODE_DEPT"
    df_zone = df_dept_pop
    # geojson_zone = dept_geojson
    patients_count, df_arranged = mise_en_forme_figure(gdf,df_zone,CODE_ZONE)
    df_arrange_gpd = gpd.GeoDataFrame(df_arranged, geometry="geometry")
    df_arrange_gpd.to_file("../data/zones_geographiques/departements/080424_dpt_patients_count.shp")
    # df_arranged.to_csv('../data/zones_geographiques/departements/270224_dpt_patients_count.csv', sep=";") 
    #patients_count.to_csv('../data/zones_geographiques/departements/patient_count_dpt.csv', sep=";")

if value =="EPCI":
    CODE_ZONE = "CODE_EPCI"
    df_zone = df_epci
    # geojson_zone = epci_geojson
    patients_count, df_arranged = mise_en_forme_figure(gdf,df_zone,CODE_ZONE)
    df_arranged.to_csv('../data/zones_geographiques/epci/epci_patients.csv', sep = ";") 
    patients_count.to_csv("../data/zones_geographiques/epci/patient_count_epci.csv", sep=";")


if value == 'Iris':
    CODE_ZONE = "CODE_IRIS"
    df_zone = df_iris
    # geojson_zone = iris_geojson
    patients_count, df_arranged = mise_en_forme_figure(gdf,df_zone,CODE_ZONE)
    df_arranged.to_csv('../data/zones_geographiques/iris/iris_patients_count.csv', sep=";") 
    #patients_count.to_csv('../data/zones_geographiques/iris/patients_count_iris.csv',sep=";")

if value == "Region":
    CODE_ZONE = "INSEE_REG"
    df_zone = df_region
    # geojson_zone = region_geojson
    patients_count, df_arranged = mise_en_forme_figure(gdf, df_zone, CODE_ZONE)
    df_arrange_gpd = gpd.GeoDataFrame(df_arranged, geometry="geometry")
    # df_arrange_gpd.to_file('../data/zones_geographiques/region/260224_region_patient_count.shp', sep = ";") 
    df_arranged.to_csv("../data/zones_geographiques/region/260224_region_patient_count.csv", sep=";")






C:\Users\jbocque1\AppData\Local\Temp\ipykernel_11012\2862834773.py:8: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  df_arrange_gpd.to_file("../data/zones_geographiques/departements/080424_dpt_patients_count.shp")


In [41]:
df_arranged

,INSEE_REG,NOM_M,patient_count,geometry
0,11,ILE-DE-FRANCE,44893,"POLYGON ((689486.400 6885591.700, 689488.900 6..."
1,24,CENTRE-VAL DE LOIRE,2632,"POLYGON ((604790.400 6831645.800, 604797.700 6..."
2,27,BOURGOGNE-FRANCHE-COMTE,987,"POLYGON ((880572.700 6730278.000, 880518.100 6..."
3,28,NORMANDIE,1423,"MULTIPOLYGON (((367660.000 6849504.200, 367662..."
4,32,HAUTS-DE-FRANCE,2296,"POLYGON ((686066.000 6888650.700, 685949.700 6..."
5,44,GRAND EST,910,"POLYGON ((983336.700 6755086.100, 983449.700 6..."
6,52,PAYS DE LA LOIRE,914,"MULTIPOLYGON (((293225.000 6674149.200, 293225..."
7,53,BRETAGNE,860,"MULTIPOLYGON (((134191.300 6821460.500, 134193..."
8,75,NOUVELLE-AQUITAINE,1235,"MULTIPOLYGON (((374733.100 6507110.200, 374731..."
9,76,OCCITANIE,667,"MULTIPOLYGON (((449522.300 6253186.500, 449526..."


In [28]:
gdf

,pseudo_provisoire,adresse,codepost,nom_commune_postal,x,y,score,trust_score,street,city,...,CODE_DEPT_right,NOM_DEPT,CODE_CHF,NOM_CHF,X_CHF_LIEU,Y_CHF_LIEU,X_CENTROID,Y_CENTROID,CODE_REG,NOM_REG
0,1.0,34 RUE DES FRERES CHAUSSONS,92600.0,ASNIERES-SUR-SEINE,2.289499,48.916298,0.848101,middle,34 Rue des Frères Chausson,Asnières-sur-Seine,...,39,JURA,300,LONS-LE-SAUNIER,895198,6622537,886172,6641548,27,BOURGOGNE-FRANCHE-COMTE
1,2.0,11 RUE EMILE DUBOIS,75014.0,PARIS,2.336628,48.831707,0.881658,middle,11 Rue Emile Dubois,Paris,...,39,JURA,300,LONS-LE-SAUNIER,895198,6622537,886172,6641548,27,BOURGOGNE-FRANCHE-COMTE
2,3.0,48 CHEMIN VERT,78680.0,EPONE,3.518424,50.329770,0.876859,middle,48 Chemin Vert,Aulnoy-lez-Valenciennes,...,39,JURA,300,LONS-LE-SAUNIER,895198,6622537,886172,6641548,27,BOURGOGNE-FRANCHE-COMTE
3,4.0,18 ALLEE DE LA CHARNILLE,47140.0,SAINT-SYLVESTRE-SUR-LOT,2.605893,48.786170,0.747258,middle,18 Allée de la Charmille,Pontault-Combault,...,39,JURA,300,LONS-LE-SAUNIER,895198,6622537,886172,6641548,27,BOURGOGNE-FRANCHE-COMTE
4,5.0,31 RUE DU GENERAL DE MIRIBEL,92500.0,RUEIL-MALMAISON,2.173326,48.865232,0.882703,middle,31 Rue du Général de Miribel,Rueil-Malmaison,...,39,JURA,300,LONS-LE-SAUNIER,895198,6622537,886172,6641548,27,BOURGOGNE-FRANCHE-COMTE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59671,64290.0,3 AV DE FOUILLEUSE,92210,SAINT-CLOUD,2.211229,48.861208,0.648836,middle,3 Avenue de Fouilleuse,Saint-Cloud,...,39,JURA,300,LONS-LE-SAUNIER,895198,6622537,886172,6641548,27,BOURGOGNE-FRANCHE-COMTE
59672,64291.0,81 COTE DU TORCHON,27220,L'HABIT,1.365413,48.871685,0.855532,middle,81 Cote du Torchon,L Habit,...,39,JURA,300,LONS-LE-SAUNIER,895198,6622537,886172,6641548,27,BOURGOGNE-FRANCHE-COMTE
59673,64292.0,60 RUE BAUDRICOURT,75013,Paris 13,2.362960,48.825882,0.887940,middle,60 Rue Baudricourt,Paris,...,39,JURA,300,LONS-LE-SAUNIER,895198,6622537,886172,6641548,27,BOURGOGNE-FRANCHE-COMTE
59674,64293.0,159 AVENUE DE LA REPUBLIQUE,92320,CHATILLON,-0.608101,44.839611,0.894598,middle,159 Avenue de la République,Bordeaux,...,39,JURA,300,LONS-LE-SAUNIER,895198,6622537,886172,6641548,27,BOURGOGNE-FRANCHE-COMTE


In [24]:
#récup liste des départements par région : 

dept_par_region = gdf.groupby('INSEE_REG')['CODE_DEPT'].unique()

dept_par_region = dept_par_region.to_dict()



{11: array(['92', '75', '77', '78', '95', '93', '91', '94'], dtype=object),
 24: array(['45', '28', '37', '41', '18', '36', '78'], dtype=object),
 27: array(['89', '25', '58', '21', '39', '71', '70', '90'], dtype=object),
 28: array(['27', '76', '14', '50', '61', nan, '95', '28'], dtype=object),
 32: array(['59', '60', '62', '80', '02', nan], dtype=object),
 44: array(['51', '54', '57', '10', '67', '08', '68', '52', '88', '55'],
       dtype=object),
 52: array(['44', '49', '72', '85', '53', nan], dtype=object),
 53: array(['29', '56', '22', '35', nan], dtype=object),
 75: array(['87', '33', '19', '16', '64', '17', '24', '79', '86', '47', '40',
        '23'], dtype=object),
 76: array(['30', '09', '66', '46', '31', '81', '34', '11', '48', '32', '82',
        '12', '65'], dtype=object),
 84: array(['74', '38', '63', '69', '42', '01', '15', '26', '03', '73', '07',
        '43'], dtype=object),
 93: array(['13', '06', '83', '05', nan, '84', '04'], dtype=object),
 94: array(['2A', '2B'], d

In [47]:
dept_par_region[75]

df_pat_region = pd.read_csv("H:/canc_air/data/zones_geographiques/region/region_patient_lib.csv", sep = ";")


reg = df_pat_region.loc[df_pat_region["lib"]=='CORSE', 'INSEE_REG'].iloc[0]

li = dept_par_region[reg].tolist()


gdf_reg = gdf[gdf["CODE_DEPT"].isin(li)]


In [48]:
gdf_reg

,pseudo_provisoire,adresse,codepost,nom_commune_postal,x,y,score,trust_score,street,city,...,code_dept,dept,reg,address;,geometry,CODE_IRIS,CODE_EPCI,CODE_DEPT,index_right,INSEE_REG
101,106.0,TOZZA ALTA 58 TRAVU,20240.0,VENTISERI,9.056618,41.496678,0.483381,middle,Strada di Tozza-Alta,Pianottoli-Caldarello,...,2A,Corse-du-Sud,Corse,Strada di Tozza-Alta 20131 Pianottoli-Caldarello;,POINT (9.05662 41.49668),2A2150000,200040764.0,2A,12.0,94
665,702.0,RESIDENCE ALZELLO CHEMIN ALZE...,20220.0,L ILE ROUSSE,8.937492,42.624728,0.365340,low,Chemin d Alzello,L Ãle-Rousse,...,2B,Haute-Corse,Corse,Chemin d Alzello 20220 L Ãle-Rousse;,POINT (8.93749 42.62473),2B1340000,200073104.0,2B,12.0,94
1314,1396.0,1 AVENUE DE ROQUENCOURT BATIMENT B ...,78150.0,LE CHESNAY,8.729399,41.919359,0.317502,low,Residence les Aloes Batiment B,Ajaccio,...,2A,Corse-du-Sud,Corse,Residence les Aloes Batiment B 20000 Ajaccio;,POINT (8.72940 41.91936),2A0040301,242010056.0,2A,12.0,94
1446,1535.0,6 RUE JEAN NICOLI,95150.0,TAVERNY,9.276955,41.590342,0.870485,middle,6 Rue Jean Nicoli,Porto-Vecchio,...,2A,Corse-du-Sud,Corse,6 Rue Jean Nicoli 20137 Porto-Vecchio;,POINT (9.27695 41.59034),2A2470101,200040764.0,2A,12.0,94
1525,1619.0,223 ALLE D'AQUITAINE APP 136 - BT...,92000.0,NANTERRE,8.729399,41.919359,0.137148,low,Residence les Aloes Batiment B,Ajaccio,...,2A,Corse-du-Sud,Corse,Residence les Aloes Batiment B 20000 Ajaccio;,POINT (8.72940 41.91936),2A0040301,242010056.0,2A,12.0,94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59229,63781.0,19 RUE CENSIER HALL B ...,75005,Paris 05,8.729399,41.919359,0.163201,low,Residence les Aloes Batiment B,Ajaccio,...,2A,Corse-du-Sud,Corse,Residence les Aloes Batiment B 20000 Ajaccio;,POINT (8.72940 41.91936),2A0040301,242010056.0,2A,12.0,94
59374,63938.0,RESIDENCE PIANA D OCCI MARINE SANT...,20260,LUMIO,8.825576,42.602536,0.477325,middle,Marine de Sant Ambroggio,Lumio,...,2B,Haute-Corse,Corse,Marine de Sant Ambroggio 20260 Lumio;,POINT (8.82558 42.60254),2B1500000,242020105.0,2B,12.0,94
59395,63960.0,255 ROUTE DE FEMINICCIA,20240,GHISONACCIA,9.412032,42.020835,0.327815,low,Strada di a Feminiccia,Ghisonaccia,...,2B,Haute-Corse,Corse,Strada di a Feminiccia 20240 Ghisonaccia;,POINT (9.41203 42.02083),2B1230000,200033827.0,2B,12.0,94
59633,64225.0,5 ROUTE DE PROPRIANO,20112,SAINTE-LUCIE-DE-TALLANO,8.924307,41.655123,0.775698,middle,Route de Propriano,SartÃ¨ne,...,2A,Corse-du-Sud,Corse,Route de Propriano 20100 SartÃ¨ne;,POINT (8.92431 41.65512),2A2720000,242010130.0,2A,12.0,94
